# WM-811K — export of EfficientNet-B2 predictions under the transformation group

This notebook loads the fitted EfficientNet-B2 model and stores its predicted
probabilities for every held-out wafer under every element of the dihedral group
$D_4$. The conformal analysis is carried out in a separate notebook, which reads
these arrays.

Splitting the work this way is what makes the study cheap. The classifier is
fitted once and held fixed, so the eight forward passes are done once here; the
conformal notebook then resamples calibration/test splits of an array already in
memory. Recomputing the network inside the replication loop would multiply the
cost by the number of replications for no gain.

**The transformation group.** $D_4$ consists of the four rotations by multiples
of 90 degrees and the four reflections obtained by composing them with a
horizontal flip. The wafer maps are square, so every element maps the pixel grid
onto itself exactly: no interpolation, no padding, no loss at the borders. The
group has eight elements, so the orbit average is computed exactly rather than
approximated by sampling.

Invariance is only approximate here, and deliberately so. Edge-Ring, Center,
Donut, Random and Near-full are defined by radial structure or by the absence of
directional structure, so a rotation or reflection plausibly preserves the
class; Edge-Loc, Loc and Scratch are defined by a linear trace or a localised
region, and for those the class-conditional distribution of orientations need
not be invariant. Conformal validity does not require invariance, so the case
study tests whether the departure shows up in efficiency rather than in
coverage.

## 1. Setup

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA PyTorch:", torch.version.cuda)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Architetture supportate:", torch.cuda.get_arch_list())

PyTorch: 2.10.0+cu128
CUDA PyTorch: 12.8
CUDA disponibile: True
GPU: Tesla T4
Compute capability: (7, 5)
Architetture supportate: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


## 2. Libraries

In [2]:
import sys
import os
import glob

# Cerca general_utils.py nei dataset collegati al notebook
matches = glob.glob(
    "/kaggle/input/**/general_utils.py",
    recursive=True
)

print("File trovati:", matches)

if not matches:
    raise FileNotFoundError(
        "general_utils.py non trovato sotto /kaggle/input"
    )

UTILS_DIR = os.path.dirname(matches[0])
print("Cartella delle utilities:", UTILS_DIR)

if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)

from general_utils import (
    image_path_generation,
    get_model_probs,
    evaluate_model
)

from effnet_utils import *

print("Utilities importate correttamente.")

File trovati: ['/kaggle/input/datasets/alaurenzi/wm811k-utils/general_utils.py']
Cartella delle utilities: /kaggle/input/datasets/alaurenzi/wm811k-utils
Utilities importate correttamente.


In [3]:
import os
import sys
import copy
import glob
import json
import math
import time
import random
import datetime
import warnings
from contextlib import contextmanager

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from albumentations import Compose, Normalize, Resize
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

# Locate the utilities among the datasets attached to the notebook.
matches = glob.glob("/kaggle/input/**/general_utils.py", recursive=True)
if not matches:
    raise FileNotFoundError("general_utils.py non trovato sotto /kaggle/input")
UTILS_DIR = os.path.dirname(matches[0])
if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)
print("Cartella delle utilities:", UTILS_DIR)

from general_utils import image_path_generation, get_model_probs

# No `from effnet_utils import *`. It redefines get_transforms, TrainDataset and
# the configuration; re-running it after section 3 silently replaces the
# definitions below, which is what produced "Input height (404) doesn't match
# model (224)". Import names from it explicitly if they are ever needed.

Cartella delle utilities: /kaggle/input/datasets/alaurenzi/wm811k-utils


## 3. Configuration and model utilities

Taken unchanged from the notebooks that fitted the model, so that the
preprocessing applied here is exactly the one the network was trained with.

In [4]:
# ====================================================
# Configuration
# ====================================================

class CFG:
    print_freq=100
    num_workers = 4
    model_name = 'tf_efficientnet_b0_ns'   #'tf_efficientnet_b2_ns' #'vgg16' #'resnext50_32x4d' #'tf_efficientnet_l2_ns_475'  #'tf_efficientnet_b2_ns' #'resnext50_32x4d'  #'coat_tiny'   
    #output_dir = '/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/WMDD_application/models'
    #intervalplot_dir = '/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/WMDD_application/resuls'
    output_dir='./'
    intervalplot_dir = './'
    size = 101 #712
    epochs = 25 # 100
    factor = 0.2
    patience = 3
    eps = 1e-6
    lr = 1e-4
    min_lr = 1e-6
    batch_size = 16
    weight_decay = 1e-6
    gradient_accumulation_steps = 1
    max_grad_norm = 1000
    seed = 42
    target_size = 8
    target_col = 'labels'
    n_fold = 3
    #trn_fold = [1,2,3,4,5]
    trn_fold = [0,1,2]
    score_plot = True
    img_size = 224
    #device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    label2int = {
    'Center': 0, 'Donut': 1, 'Edge-Loc': 2, 'Edge-Ring': 3,
    'Loc': 4, 'Random': 5, 'Scratch': 6, 'Near-full': 7
                                                        }
    randomize = True, #for  get_APS_scores_all function
    seed = 0         #for  get_APS_scores_all function
    n_sim = 1000
    alpha=0.01
    score = 'APS'  # method to compute scores ('APS', 'RAPS', 'STD')
    numclasses = 8

    # ------------------------------------------------------------------
    # Export: every model in `models` is processed in turn.
    # A checkpoint f'{name}_best.pth' must exist in one of `ckpt_dirs`.
    # ------------------------------------------------------------------
    models = [ model_name]
    ckpt_dirs = ['/kaggle/input/datasets/alaurenzi/wm81k-effnet-model']
    img_size_override = {}        # {name: size} for a model trained at another size
    file_tag = {'tf_efficientnet_b2_ns': 'efficientnet_b2'}   # names already read by the conformal notebook
    out_dir = '/kaggle/working/csda_revision/data'
    export_workers = 2            # DataLoader workers; 0 if worker processes cause trouble
    overwrite = False             # False: models already exported in out_dir are reloaded, not recomputed

cfg=CFG()

In [5]:
# ====================================================
# Output and Logging
# ====================================================
#cfg.output_dir = './'

def init_logger(log_file=cfg.output_dir + 'train.log'):
    from logging import getLogger, INFO, FileHandler, Formatter, StreamHandler
    logger = getLogger(__name__)
    logger.setLevel(INFO)
    handler1 = StreamHandler()
    handler1.setFormatter(Formatter("%(message)s"))
    handler2 = FileHandler(filename=log_file)
    handler2.setFormatter(Formatter("%(message)s"))
    logger.addHandler(handler1)
    logger.addHandler(handler2)
    return logger
LOGGER = init_logger()

@contextmanager
def timer(name):
    t0 = time.time()
    LOGGER.info(f'[{name}] start')
    yield
    LOGGER.info(f'[{name}] done in {time.time() - t0:.0f} s.')

# ====================================================
# Utility Functions
# ====================================================
def plot_score(score_list):
    plt.plot(list(range(len(score_list))), score_list, '-o')
    plt.title('Score over epochs')
    plt.xlabel('Epochs')
    plt.ylabel('AUC Score')
    plt.xticks(list(range(len(score_list))))
    plt.grid(True)
    plt.show()

def get_score(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def get_auc(y_true, y_pred):
    return roc_auc_score(y_true, y_pred, multi_class='ovr', average='macro')

def seed_torch(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# ====================================================
# Helper Classes
# ====================================================
class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return f'{m}m {s:.0f}s'

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / percent
    rs = es - s
    return f'{asMinutes(s)} (remain {asMinutes(rs)})'

# ====================================================
# Dataset and Transforms
# ====================================================
class TrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.file_names = df['image_id'].values
        self.labels = df['labels'].values
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = cv2.imread(self.file_names[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        label = torch.tensor(self.labels[idx]).long()
        return image, label

def get_transforms(data, cfg):
    transform_list = [
        Resize(cfg.img_size, cfg.img_size),
        Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]
    return Compose(transform_list)

# ====================================================
# Training and Validation Functions
# ====================================================
def train_fn(train_loader, model, criterion, optimizer, epoch, scheduler, device, cfg):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()

    model.train()
    start = end = time.time()
    global_step = 0

    for step, (images, labels) in enumerate(train_loader):
        data_time.update(time.time() - end)
        images, labels = images.to(device), labels.to(device)
        batch_size = labels.size(0)

        y_preds = model(images)
        loss = criterion(y_preds, labels)
        losses.update(loss.item(), batch_size)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)

        if (step + 1) % cfg.gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        batch_time.update(time.time() - end)
        end = time.time()

        if step % cfg.print_freq == 0 or step == (len(train_loader) - 1):
            print(f'Epoch: [{epoch+1}][{step}/{len(train_loader)}] '
                  f'Data {data_time.val:.3f} ({data_time.avg:.3f}) '
                  f'Elapsed {timeSince(start, (step+1)/len(train_loader))} '
                  f'Loss: {losses.val:.4f}({losses.avg:.4f})')

    return losses.avg

def valid_fn(valid_loader, model, criterion, device):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()

    model.eval()
    preds = []
    start = end = time.time()

    for step, (images, labels) in enumerate(valid_loader):
        data_time.update(time.time() - end)
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            y_preds = model(images)

        loss = criterion(y_preds, labels)
        losses.update(loss.item(), labels.size(0))
        preds.append(y_preds.softmax(1).cpu().numpy())

        batch_time.update(time.time() - end)
        end = time.time()

        if step % CFG.print_freq == 0 or step == (len(valid_loader) - 1):
            print(f'EVAL: [{step}/{len(valid_loader)}] '
                  f'Data {data_time.val:.3f} ({data_time.avg:.3f}) '
                  f'Elapsed {timeSince(start, (step+1)/len(valid_loader))} '
                  f'Loss: {losses.val:.4f}({losses.avg:.4f})')

    return losses.avg, np.concatenate(preds)

# ====================================================
# Train Loop
# ====================================================
def train_loop(train, val, cfg):
    LOGGER.info("========== start: training ==========")

    train_dataset = TrainDataset(train.reset_index(drop=True), transform=get_transforms(data='train', cfg=cfg))
    valid_dataset = TrainDataset(val.reset_index(drop=True), transform=get_transforms(data='valid', cfg=cfg))

    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=False)

    model = Model_type(cfg.model_name, cfg=cfg, pretrained=True).to(device)
    optimizer = Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=cfg.factor,
                                  patience=cfg.patience, verbose=True, eps=cfg.eps)
    criterion = nn.CrossEntropyLoss()

    best_score = 0.
    best_loss = np.inf
    score_list = []
    patience_count = 0
    last_score = 0

    for epoch in range(cfg.epochs):
        start_time = time.time()
        avg_loss = train_fn(train_loader, model, criterion, optimizer, epoch, scheduler, device, cfg)
        avg_val_loss, preds = valid_fn(valid_loader, model, criterion, device)

        valid_labels = val[cfg.target_col].values
        scheduler.step(avg_val_loss)

        accuracy = get_score(valid_labels, preds.argmax(1))
        val_pred = preds / preds.sum(axis=1)[:, None]
        score = get_auc(valid_labels, val_pred)

        elapsed = time.time() - start_time
        LOGGER.info(f'Epoch {epoch+1} - avg_train_loss: {avg_loss:.4f}  avg_val_loss: {avg_val_loss:.4f}  time: {elapsed:.0f}s')
        LOGGER.info(f'Epoch {epoch+1} - Accuracy: {accuracy:.4f} AUC: {score:.4f}')

        if score > best_score:
            best_score = score
            LOGGER.info(f'Epoch {epoch+1} - Save Best Score: {best_score:.4f} Model')
            torch.save({'model': model.state_dict(), 'preds': preds}, OUTPUT_DIR + f'{cfg.model_name}_best.pth')

        score_list.append(score)
        if score > last_score:
            last_score = score
            patience_count = 0
        else:
            patience_count += 1
            if patience_count > cfg.patience:
                LOGGER.info(f'***Max patience {cfg.patience} reached - AUC: {score:.4f}***')
                break

    if cfg.score_plot:
        plot_score(score_list)

    check_point = torch.load(OUTPUT_DIR + f'{cfg.model_name}_best.pth', weights_only=False)
    val['preds'] = check_point['preds'].argmax(1)

    return val

# ====================================================
# Main Function
# ====================================================
def main(cfg, train, val):
    def get_result(result_df):
        preds = result_df['preds'].values
        labels = result_df[cfg.target_col].values
        score = get_score(labels, preds)
        LOGGER.info(f'Score: {score:<.5f}')

    oof_df = pd.DataFrame()

    _oof_df = train_loop(train, val, cfg)
    oof_df = pd.concat([oof_df, _oof_df], ignore_index=True)

    get_result(_oof_df)
    LOGGER.info(f"========== CV ==========")
    get_result(oof_df)

    oof_df.to_csv(OUTPUT_DIR + 'oof_df.csv', index=False)


# ====================================================
# Model Definition
# ==================================================== resnext50_32x4d_best.pth
class Model_type(nn.Module):
    #def __init__(self, model_name='tf_efficientnet_b2_ns', pretrained=False, cfg=None):
    #    super().__init__()
    #def __init__(self, model_name='resnext50_32x4d_best.pth', pretrained=False, cfg=None):
     #   super().__init__()
    def __init__(self, model_name=cfg.model_name, pretrained=False, cfg=None):
        super().__init__()

        # If cfg is not provided, fallback to a default
        if cfg is None:
            cfg = CFG()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=cfg.target_size)


    def forward(self, x):
        x = self.model(x)
        return x

# Function to compute and compare evaluation metrics
def evaluate_model(y_true, y_pred, y_probs):
    # Compute the metrics
    accuracy = accuracy_score(y_true, y_pred)
    macro_recall = recall_score(y_true, y_pred, average='macro')
    roc_auc = roc_auc_score(y_true, y_probs, multi_class='ovr')

    print(f'Overall Accuracy: {accuracy:.5f}')
    print(f'Macro-averaged recall score: {macro_recall:.5f}')

## 4. The transformation group

The group element is applied to the raw image, before normalisation, so that
the transformation and the preprocessing do not interact.

In [6]:
# ---------------------------------------------------------------------------
# Dihedral group D4 applied inside the dataset pipeline.
#
# The transformation acts on the raw image, before normalisation, so that the
# group element and the preprocessing do not interact. The images are square
# (cfg.size = 101), so rotations by multiples of 90 degrees map the pixel grid
# onto itself exactly: no interpolation, no padding, no loss at the borders.
# ---------------------------------------------------------------------------

from image_groups import DihedralGroup

GROUP = DihedralGroup()               # 8 elements; the full orbit is enumerable

class TransformedDataset(TrainDataset):
    """TrainDataset with one fixed group element applied to every image."""

    def __init__(self, df, g, transform=None):
        super().__init__(df, transform)
        self.g = int(g)

    def __getitem__(self, idx):
        image = cv2.imread(self.file_names[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # apply expects a batch, so add and remove the leading axis
        image = GROUP.apply(image[None, ...], self.g)[0]
        if self.transform:
            image = self.transform(image=image)['image']
        return image, torch.tensor(self.labels[idx]).long()


def orbit_probs(model, df, device, cfg, group=GROUP):
    """Predicted probabilities under every group element: shape (|G|, n, K).

    Computed once and saved. The conformal notebook then resamples
    calibration/test splits of an array already in memory; recomputing the
    network inside the replication loop would multiply the cost by R for
    nothing, since the model is fixed.
    """
    out = []
    for g in group.full_orbit():
        ds = TransformedDataset(df, g, transform=get_transforms(data='valid', cfg=cfg))
        dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.export_workers, pin_memory=True)
        out.append(get_model_probs(model, dl, device))
    return np.stack(out)


def plain_probs(model, df, device, cfg):
    """Predicted probabilities with no group element applied: shape (n, K)."""
    ds = TrainDataset(df, transform=get_transforms(data='valid', cfg=cfg))
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                    num_workers=cfg.export_workers, pin_memory=True)
    return get_model_probs(model, dl, device)

## 5. Data

Calibration and test are merged into a single held-out set. The conformal
analysis resamples the calibration/test split at every replication, so the
variability of calibration is measured rather than fixed by one arbitrary
split. The training and validation images are not touched: the model must not
have seen any wafer used for calibration or evaluation.

In [7]:
#dataset_path = f'{BASE}/WMDD_application/dataset/'
dataset_path = '/kaggle/input/datasets/alaurenzi/wm811k-images-dataset/images/'

df = pd.read_csv(dataset_path + 'WM_811k_subset.csv', index_col=0)
df['image_id'] = df.apply(lambda row: image_path_generation(row, base_path=dataset_path), axis=1)

heldout = (df[df['set'].isin(['cal', 'test'])][['labels', 'image_id']]
             .reset_index(drop=True))
heldout['labels'] = heldout['labels'].map(cfg.label2int)

assert heldout['labels'].notna().all(), "some label is missing from cfg.label2int"
heldout['labels'] = heldout['labels'].astype(int)

INT2LABEL = {v: k for k, v in cfg.label2int.items()}
print('held-out size:', heldout.shape)
display(heldout['labels'].map(INT2LABEL).value_counts().rename('wafers').to_frame().T)

held-out size: (3649, 2)


labels,Edge-Ring,Edge-Loc,Center,Loc,Random,Scratch,Donut,Near-full
wafers,1389,635,615,413,203,202,148,44


## 6. Model

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def tag_of(name):
    """File prefix of a model in cfg.out_dir."""
    return cfg.file_tag.get(name, name)


def model_cfg(name, base=cfg):
    """Copy of the configuration with the model-specific fields set."""
    c = copy.copy(base)
    c.model_name = name
    c.img_size = base.img_size_override.get(name, base.img_size)
    return c


def find_checkpoint(name, dirs):
    hits = [os.path.join(d, f'{name}_best.pth') for d in dirs]
    hits = [p for p in hits if os.path.isfile(p)]
    if len(hits) != 1:
        raise FileNotFoundError(f'{name}: expected exactly one {name}_best.pth in {dirs}, found {hits}')
    return hits[0]


def load_model(name, c):
    state = torch.load(find_checkpoint(name, c.ckpt_dirs), map_location='cpu', weights_only=False)['model']
    state = {k.removeprefix('module.'): v for k, v in state.items()}   # DataParallel checkpoints
    model = Model_type(name, pretrained=False, cfg=c)
    model.load_state_dict(state, strict=True)
    return model.to(device).eval()


def preflight(model, c, df):
    """Input size and head size agree with the pipeline, for every group element."""
    s = c.img_size
    with torch.no_grad():
        k = model(torch.zeros(1, 3, s, s, device=device)).shape[1]
    assert k == c.numclasses, f'{c.model_name}: head has {k} outputs, expected {c.numclasses}'
    tf = get_transforms(data='valid', cfg=c)
    for g in GROUP.full_orbit():
        x, _ = TransformedDataset(df.iloc[:1], g, transform=tf)[0]
        assert tuple(x.shape) == (3, s, s), \
            f'{c.model_name}, g={g}: pipeline gives {tuple(x.shape)}, model expects (3, {s}, {s})'


# Every checkpoint must exist and every file tag must be distinct before starting.
missing = []
for name in cfg.models:
    try:
        print(f'{name:28s} -> {tag_of(name):20s} {find_checkpoint(name, cfg.ckpt_dirs)}')
    except FileNotFoundError as e:
        missing.append(str(e))
if missing:
    raise FileNotFoundError('\n'.join(missing))
tags = [tag_of(n) for n in cfg.models]
assert len(set(tags)) == len(tags), f'two models share a file tag: {tags}'
print('device:', device)

tf_efficientnet_b0_ns        -> tf_efficientnet_b0_ns /kaggle/input/datasets/alaurenzi/wm81k-effnet-model/tf_efficientnet_b0_ns_best.pth
device: cuda


## 7. Export

Three assertions guard the output. The shape must match the group size, the
number of wafers and the number of classes; the rows must be probability
vectors; and element 0 of the group, being the identity, must reproduce the
untransformed prediction. The last one is the check that the ordering of the
group elements agrees with what the conformal notebook assumes.

In [9]:
os.makedirs(cfg.out_dir, exist_ok=True)
labels = heldout['labels'].to_numpy().astype(np.int64)
PROBS = {}

for name in cfg.models:
    c = model_cfg(name)
    tag = tag_of(name)
    f_probs = f'{cfg.out_dir}/{tag}_probs.npy'
    print(f'\n=== {name} (img_size={c.img_size}) -> {tag} ===')

    if os.path.isfile(f_probs) and not cfg.overwrite:
        probs = np.load(f_probs)
        assert probs.shape == (GROUP.size, len(labels), c.numclasses), f'{f_probs}: shape {probs.shape}'
        assert np.array_equal(np.load(f'{cfg.out_dir}/{tag}_labels.npy'), labels), f'{tag}: labels differ'
        PROBS[name] = probs
        print(f'already exported, reloaded {probs.shape}')
        continue

    model = load_model(name, c)
    preflight(model, c, heldout)

    probs = orbit_probs(model, heldout, device, c)          # (|G|, n, K)
    plain = plain_probs(model, heldout, device, c)          # (n, K)

    assert probs.shape == (GROUP.size, len(labels), c.numclasses)
    assert np.allclose(probs.sum(axis=2), 1.0, atol=1e-4), "rows must sum to one"
    assert np.allclose(probs[0], plain, atol=1e-4), "element 0 is not the identity"

    acc = float((probs[0].argmax(1) == labels).mean())
    np.save(f_probs, probs.astype(np.float32))
    np.save(f'{cfg.out_dir}/{tag}_labels.npy', labels)
    with open(f'{cfg.out_dir}/{tag}_meta.json', 'w') as fh:
        json.dump({'model_name': name, 'file_tag': tag,
                   'checkpoint': find_checkpoint(name, c.ckpt_dirs),
                   'img_size': c.img_size, 'group': type(GROUP).__name__,
                   'group_size': int(GROUP.size), 'n': int(len(labels)),
                   'numclasses': int(c.numclasses), 'label2int': c.label2int,
                   'heldout_accuracy': acc, 'timm': timm.__version__,
                   'torch': torch.__version__,
                   'exported': datetime.datetime.now().isoformat(timespec='seconds')},
                  fh, indent=2)

    PROBS[name] = probs.astype(np.float32)
    print(f'saved {probs.shape} to {cfg.out_dir}; held-out accuracy {acc:.3f}')

    del model, probs, plain
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\nexported:', list(PROBS))


=== tf_efficientnet_b0_ns (img_size=224) -> tf_efficientnet_b0_ns ===


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

...

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

saved (8, 3649, 8) to /kaggle/working/csda_revision/data; held-out accuracy 0.917

exported: ['tf_efficientnet_b0_ns']


## 8. Dispersion along the orbit

How much the predicted probabilities move as the group element varies. The
theory says orbit averaging helps to the extent that the score varies along the
orbit, so this is the quantity to look at before any calibration. If the radial
classes disperse less than the directional ones, the approximate nature of the
invariance is already visible here.

In [10]:
RADIAL = ['Center', 'Donut', 'Edge-Ring', 'Random', 'Near-full']

rows = []
for name, probs in PROBS.items():
    disp = probs.std(axis=0).mean(axis=1)          # per wafer, across group elements
    for k in range(cfg.numclasses):
        m = labels == k
        cls = INT2LABEL[k]
        rows.append({'model': name, 'class': cls,
                     'symmetry': 'radial' if cls in RADIAL else 'directional',
                     'n': int(m.sum()), 'dispersion': float(disp[m].mean())})

d = pd.DataFrame(rows)
display(d.pivot_table(index=['symmetry', 'class', 'n'], columns='model',
                      values='dispersion').round(4))
display(d.groupby(['symmetry', 'model'])['dispersion'].mean()
         .unstack('model').round(4))

model                       tf_efficientnet_b0_ns
symmetry    class     n                          
directional Edge-Loc  635                  0.0309
            Loc       413                  0.0596
            Scratch   202                  0.0349
radial      Center    615                  0.0192
            Donut     148                  0.0231
            Edge-Ring 1389                 0.0054
            Near-full 44                   0.0056
            Random    203                  0.0163

model,tf_efficientnet_b0_ns
symmetry,
directional,0.0418
radial,0.0139


In [11]:
RADIAL = ['Center', 'Donut', 'Edge-Ring', 'Random', 'Near-full']

disp = probs.std(axis=0).mean(axis=1)          # per wafer, across group elements
rows = []
for c in range(cfg.numclasses):
    m = labels == c
    name = INT2LABEL[c]
    rows.append({'class': name,
                 'symmetry': 'radial' if name in RADIAL else 'directional',
                 'n': int(m.sum()),
                 'dispersion': float(disp[m].mean())})

d = pd.DataFrame(rows).sort_values(['symmetry', 'dispersion'])
display(d.round(4))
display(d.groupby('symmetry')['dispersion'].mean().round(4).to_frame('mean'))

,class,symmetry,n,dispersion
2,Edge-Loc,directional,635,0.0309
6,Scratch,directional,202,0.0349
4,Loc,directional,413,0.0596
3,Edge-Ring,radial,1389,0.0054
7,Near-full,radial,44,0.0056
5,Random,radial,203,0.0163
0,Center,radial,615,0.0192
1,Donut,radial,148,0.0231


,mean
symmetry,
directional,0.0418
radial,0.0139


## 9. Sanity check on the fitted model

Accuracy and macro recall on the held-out set, from the untransformed
predictions. Reported to confirm that the loaded weights reproduce the
performance recorded when the model was fitted.

In [12]:
rows = []
for name, probs in PROBS.items():
    p = probs[0]
    yhat = p.argmax(axis=1)
    rows.append({'model': name,
                 'accuracy': accuracy_score(labels, yhat),
                 'macro_recall': recall_score(labels, yhat, average='macro'),
                 'auc_ovr': roc_auc_score(labels, p, multi_class='ovr'),
                 'accuracy_orbit_avg': accuracy_score(labels, probs.mean(axis=0).argmax(axis=1))})

display(pd.DataFrame(rows).set_index('model').round(5))

,accuracy,macro_recall,auc_ovr,accuracy_orbit_avg
model,,,,
tf_efficientnet_b0_ns,0.91696,0.90444,0.99258,0.93368


In [13]:
evaluate_model(labels, probs[0].argmax(axis=1), probs[0])

Overall Accuracy: 0.91696
Macro-averaged recall score: 0.90444
